# HERMES blinded visual-support review

Upload this notebook to the same AutoDL instance and run all cells in CPU/no-GPU mode.
It uses the existing target-binding video cache. This notebook needs neither a fresh
checkout nor a model, dataset download, or GPU. It checks the clip hashes and saved
decoder timestamps before rendering 147 windows across 32 questions.

Download **HERMES_blind_support_review.zip**, unzip it, and open `review.html`.
The worksheet uses the same window IDs. Record the supported option, the sampled
frame IDs in the notes, and whether the evidence applies at the question time.
If the evidence is unreadable or ambiguous, mark the window as uncertain.
Each reviewer should use a separate copy of the worksheet.


In [ ]:
from pathlib import Path
import base64, csv, hashlib, os, shutil, subprocess, sys
from zipfile import ZipFile

BASE = Path('/root/autodl-tmp')
MEDIA = BASE / 'hermes_target_binding/videos'
WORK = BASE / 'hermes_blind_support_review'
VALIDATE_ONLY = os.environ.get('HERMES_BLIND_REVIEW_VALIDATE_ONLY') == '1'
OUTPUT_NAME = 'HERMES_blind_support_review.zip'


In [ ]:
PAYLOAD = {'visual_support_review.csv': ('...'), 'review_reference.zip': ('...'), 'render_target_binding_review.py': ('IiIiQ3JlYXRlIGEgQ1BVLW9ubHkgYmxpbmQgcmV2aWV3IFpJUCBmcm9tIHRoZSBvcmlnaW5hbCB0YXJnZXQtYmluZGluZyB2aWRlb3MuIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBodG1sCmltcG9ydCBqc29uCmltcG9ydCByZQppbXBvcnQgdGVtcGZpbGUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gemlwZmlsZSBpbXBvcnQgWklQX0RFRkxBVEVELCBaaXBGaWxlCgpmcm9tIGF1dG9kbF9jb25maXJtYXRpb25fdmlzdWFsX3JldmlldyBpbXBvcnQgcmVhZGVyX2ZvciwgcmVuZGVyX3NoZWV0LCB2ZXJpZnlfdGltZXN0YW1wcwoKCmRlZiByZXZpZXdfaW5kZXgocm93cyk6CiAgICBieV9xdWVzdGlvbiA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgYnlfcXVlc3Rpb25bcm93WyJyZXZpZXdfaWQiXS5zcGxpdCgiLSIpWzBdXS5hcHBlbmQocm93KQogICAgbGluZXMgPSBbJzwhZG9jdHlwZSBodG1sPjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij48dGl0bGU+SEVSTUVTIGJsaW5kZWQgcmV2aWV3PC90aXRsZT4nLAogICAgICAgICAgICAgJzxzdHlsZT5ib2R5e2ZvbnQ6MTZweCBzeXN0ZW0tdWk7bWF4LXdpZHRoOjk2MHB4O21hcmdpbjoyZW0gYXV0bztwYWRkaW5nOjAgMWVtfScKICAgICAgICAgICAgICdpbWd7bWF4LXdpZHRoOjEwMCU7aGVpZ2h0OmF1dG99c3VtbWFyeXtjdXJzb3I6cG9pbnRlcjtwYWRkaW5nOi41ZW19c2VjdGlvbnttYXJnaW4tYm90dG9tOjJlbX08L3N0eWxlPicsCiAgICAgICAgICAgICAnPGgxPkhFUk1FUyBibGluZGVkIHZpc3VhbC1zdXBwb3J0IHJldmlldzwvaDE+JywKICAgICAgICAgICAgICc8cD5Vc2UgdGhlIGFjY29tcGFueWluZyBDU1YgdG8gcmVjb3JkIHRoZSBzdXBwb3J0ZWQgb3B0aW9uLCBzdXBwb3J0aW5nIHNhbXBsZWQgaW5kaWNlcywgJwogICAgICAgICAgICAgJ2FuZCB3aGV0aGVyIHRoZSBldmlkZW5jZSBhcHBsaWVzIGF0IHRoZSBxdWVzdGlvbiB0aW1lLiBNYXJrIHVuY2VydGFpbiBjYXNlcyBleHBsaWNpdGx5LiAnCiAgICAgICAgICAgICAnRWFjaCByZXZpZXdlciBzaG91bGQga2VlcCB0aGVpciBvd24gd29ya3NoZWV0IGNvcHkuPC9wPiddCiAgICBmb3IgcWlkLCBncm91cCBpbiBieV9xdWVzdGlvbi5pdGVtcygpOgogICAgICAgIHF1ZXN0aW9uID0gZ3JvdXBbMF0KICAgICAgICBsaW5lcy5hcHBlbmQoZiI8c2VjdGlvbj48aDI+e3FpZH0gwrcge2Zsb2F0KHF1ZXN0aW9uWydxdWVyeV90aW1lX3MnXSk6Z30gczwvaDI+IgogICAgICAgICAgICAgICAgICAgICBmIjxwPntodG1sLmVzY2FwZShxdWVzdGlvblsncXVlc3Rpb24nXSl9PC9wPjxvbCB0eXBlPSdBJz4iKQogICAgICAgIGxpbmVzLmV4dGVuZChmIjxsaT57aHRtbC5lc2NhcGUob3B0aW9uKX08L2xpPiIgZm9yIG9wdGlvbiBpbiBqc29uLmxvYWRzKHF1ZXN0aW9uWyJvcHRpb25zIl0pKQogICAgICAgIGxpbmVzLmFwcGVuZCgiPC9vbD4iKQogICAgICAgIGZvciByb3cgaW4gZ3JvdXA6CiAgICAgICAgICAgIHJpZCA9IHJvd1sicmV2aWV3X2lkIl0KICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYnPGRldGFpbHM+PHN1bW1hcnk+e3JpZH08L3N1bW1hcnk+JwogICAgICAgICAgICAgICAgICAgICAgICAgZic8YSBocmVmPSJmcmFtZXMve3JpZH0uanBnIj48aW1nIGxvYWRpbmc9ImxhenkiIHNyYz0iZnJhbWVzL3tyaWR9LmpwZyIgYWx0PSJ7cmlkfSI+PC9hPjwvZGV0YWlscz4nKQogICAgICAgIGxpbmVzLmFwcGVuZCgiPC9zZWN0aW9uPiIpCiAgICByZXR1cm4gIlxuIi5qb2luKGxpbmVzKQoKCmRlZiBidWlsZF9wYWNrZXQod29ya3NoZWV0LCBldmlkZW5jZSwgdmlkZW9zLCBvdXRwdXQpOgogICAgd2l0aCB3b3Jrc2hlZXQub3BlbihlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBoYW5kbGU6CiAgICAgICAgcm93cyA9IGxpc3QoY3N2LkRpY3RSZWFkZXIoaGFuZGxlKSkKICAgIHdpdGggWmlwRmlsZShldmlkZW5jZSkgYXMgYXJjaGl2ZToKICAgICAgICBhc3NldHMgPSB7YVsic2FtcGxlX2lkIl06IGEgZm9yIGEgaW4ganNvbi5sb2FkcyhhcmNoaXZlLnJlYWQoImFzc2V0X21hbmlmZXN0Lmpzb24iKSlbInZpZGVvcyJdfQogICAgICAgIHRpbWVzID0ge3NpZDoganNvbi5sb2FkcyhhcmNoaXZlLnJlYWQoZiJwcm92ZW5hbmNlL2ZlYXR1cmVzX3tzaWR9Lmpzb24iKSlbInRpbWVzdGFtcHMiXSBmb3Igc2lkIGluIGFzc2V0c30KICAgIGdyb3VwZWQgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgaWRzID0ge3Jvd1sicmV2aWV3X2lkIl0gZm9yIHJvdyBpbiByb3dzfQogICAgaWYgbGVuKGlkcykgIT0gbGVuKHJvd3MpIG9yIG5vdCBpZHMgb3IgYW55KG5vdCByZS5mdWxsbWF0Y2gociJRXGR7Mn0tW0EtRV0iLCByaWQpIGZvciByaWQgaW4gaWRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCB1bmlxdWUgYmxpbmQgcmV2aWV3IElEcyIpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgc2lkID0gaW50KHJvd1sic2FtcGxlX2lkIl0pCiAgICAgICAgcXVlcnlfdGltZSA9IGZsb2F0KHJvd1sicXVlcnlfdGltZV9zIl0pCiAgICAgICAgaW5kaWNlcyA9IGpzb24ubG9hZHMocm93WyJzYW1wbGVkX2luZGljZXMiXSkKICAgICAgICBpZiBsZW4oaW5kaWNlcykgIT0gNCBvciBzb3J0ZWQoc2V0KGluZGljZXMpKSAhPSBpbmRpY2VzIG9yIGluZGljZXNbMF0gPCAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgZm91ciBvcmRlcmVkIGZyYW1lIHNsb3RzIGZvciB7cm93WydyZXZpZXdfaWQnXX0iKQogICAgICAgIGZyYW1lcyA9IFt0aW1lc1tzaWRdW2ldIGZvciBpIGluIGluZGljZXNdCiAgICAgICAgaWYgKGFueShmbG9hdChmWyJzdGFydF9zIl0pID4gcXVlcnlfdGltZSArIDFlLTYgZm9yIGYgaW4gZnJhbWVzKQogICAgICAgICAgICAgICAgb3IgW2ZbInNvdXJjZV9mcmFtZSJdIGZvciBmIGluIGZyYW1lc10gIT0ganNvbi5sb2Fkcyhyb3dbInNvdXJjZV9mcmFtZV9pbmRpY2VzIl0pCiAgICAgICAgICAgICAgICBvciBbZlsic3RhcnRfcyJdIGZvciBmIGluIGZyYW1lc10gIT0ganNvbi5sb2Fkcyhyb3dbImZyYW1lX3N0YXJ0X3MiXSkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiV29ya3NoZWV0IGRpc2FncmVlcyB3aXRoIGFyY2hpdmVkIGZyYW1lIGV2aWRlbmNlOiB7cm93WydyZXZpZXdfaWQnXX0iKQogICAgICAgIGdyb3VwZWRbc2lkXS5hcHBlbmQoKHJvd1sicmV2aWV3X2lkIl0sIHF1ZXJ5X3RpbWUsIGZyYW1lcykpCiAgICBtaXNzaW5nID0gW3NpZCBmb3Igc2lkIGluIGdyb3VwZWQgaWYgbm90ICh2aWRlb3MgLyBmInNhbXBsZV97c2lkfV9yZWFsLm1wNCIpLmlzX2ZpbGUoKV0KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIG9yaWdpbmFsIHZpZGVvcyBmb3Igc2FtcGxlIElEczoge21pc3Npbmd9IikKCiAgICBpZiBvdXRwdXQuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKGYiQ2hvb3NlIGEgbmV3IG91dHB1dCBmaWxlbmFtZToge291dHB1dH0iKQogICAgb3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeShkaXI9b3V0cHV0LnBhcmVudCkgYXMgdG1wOgogICAgICAgIHNjcmF0Y2ggPSBQYXRoKHRtcCkKICAgICAgICBkZXN0ID0gc2NyYXRjaCAvICJyZXZpZXcuemlwIgogICAgICAgIHdpdGggWmlwRmlsZShkZXN0LCAidyIsIGNvbXByZXNzaW9uPVpJUF9ERUZMQVRFRCkgYXMgYXJjaGl2ZToKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZSh3b3Jrc2hlZXQsICJ2aXN1YWxfc3VwcG9ydF9yZXZpZXcuY3N2IikKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZXN0cigicmV2aWV3Lmh0bWwiLCByZXZpZXdfaW5kZXgocm93cykpCiAgICAgICAgICAgIGZvciBzaWQsIHdpbmRvd3MgaW4gc29ydGVkKGdyb3VwZWQuaXRlbXMoKSk6CiAgICAgICAgICAgICAgICByZWFkZXIgPSByZWFkZXJfZm9yKGRpY3Qoc2FtcGxlX2lkPXNpZCwgdmlkZW9fc2hhMjU2PWFzc2V0c1tzaWRdWyJzaGEyNTYiXSksIHZpZGVvcykKICAgICAgICAgICAgICAgIGZvciByaWQsIHF1ZXJ5X3RpbWUsIGZyYW1lcyBpbiB3aW5kb3dzOgogICAgICAgICAgICAgICAgICAgIG5vcm1hbGl6ZWQgPSBbZGljdChmLCBzdGFydF9zPWZsb2F0KGZbInN0YXJ0X3MiXSksIHNvdXJjZV9mcmFtZT1pbnQoZlsic291cmNlX2ZyYW1lIl0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzYW1wbGVkX2luZGV4PWludChmWyJzYW1wbGVkX2luZGV4Il0pKSBmb3IgZiBpbiBmcmFtZXNdCiAgICAgICAgICAgICAgICAgICAgdmVyaWZ5X3RpbWVzdGFtcHMocmVhZGVyLCBub3JtYWxpemVkKQogICAgICAgICAgICAgICAgICAgIHNoZWV0ID0gc2NyYXRjaCAvIGYie3JpZH0uanBnIgogICAgICAgICAgICAgICAgICAgIHJlbmRlcl9zaGVldChyZWFkZXIsIG5vcm1hbGl6ZWQsIGYie3JpZH0gfCBRdWVzdGlvbiBhdCB7cXVlcnlfdGltZTpnfSBzIiwgMiwgNzIwLCA0MDUsIHNoZWV0KQogICAgICAgICAgICAgICAgICAgIGFyY2hpdmUud3JpdGUoc2hlZXQsIGYiZnJhbWVzL3tzaGVldC5uYW1lfSIpCiAgICAgICAgICAgICAgICBkZWwgcmVhZGVyCiAgICAgICAgICAgICAgICBwcmludChmIlJFVklFV19DTElQX1JFQURZIHNhbXBsZT17c2lkfSB3aW5kb3dzPXtsZW4od2luZG93cyl9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBkZXN0LnJlcGxhY2Uob3V0cHV0KQogICAgcHJpbnQoZiJSZW5kZXJlZCB7bGVuKHJvd3MpfSBibGluZGVkIHdpbmRvd3MgZnJvbSB7bGVuKGdyb3VwZWQpfSB2ZXJpZmllZCBjbGlwczoge291dHB1dH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3Jrc2hlZXQiLCByZXF1aXJlZD1UcnVlLCB0eXBlPVBhdGgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV2aWRlbmNlIiwgcmVxdWlyZWQ9VHJ1ZSwgdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS12aWRlb3MiLCByZXF1aXJlZD1UcnVlLCB0eXBlPVBhdGgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBidWlsZF9wYWNrZXQoYXJncy53b3Jrc2hlZXQsIGFyZ3MuZXZpZGVuY2UsIGFyZ3MudmlkZW9zLCBhcmdzLm91dHB1dCkK', 'bde7ec257bf030b760f1d7bddbfeeb01d48475fab55701948f7c683f57b100b8'), 'autodl_confirmation_visual_review.py': ('IiIiUmVuZGVyIHRoZSBleGFjdCBzYXZlZCBwcmUtcXVlcnkgZnJhbWUgc2xvdHMgZnJvbSB0aGUgdmVyaWZpZWQgbG9jYWwgY2xpcHMuIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBoYXNobGliCmltcG9ydCBodG1sCmltcG9ydCBqc29uCmltcG9ydCByZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB6aXBmaWxlIGltcG9ydCBaSVBfREVGTEFURUQsIFppcEZpbGUKCk5PVEVTX0ZJRUxEUyA9ICgKICAgICJjYXNlX2lkIiwgInJldmlld2VyIiwgInN1cHBvcnRfcmVnaW9uIiwgInN1cHBvcnRpbmdfc2FtcGxlX2luZGljZXMiLCAicmV2aWV3ZXJfY2hvaWNlIiwKICAgICJjb25maWRlbmNlIiwgImFtYmlndWl0eV9vcl9tb3Rpb24iLCAibm90ZXMiLAopClJFR0lPTlMgPSAoInJlY2VudF9vbmx5IiwgIm9sZGVyX29ubHkiLCAiYm90aF9pbmRlcGVuZGVudGx5IiwgIm5lZWRzX2JvdGgiLCAibm90X3Zpc2libGUiLCAidW5jZXJ0YWluIikKCgpkZWYgc2hhKHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIGhhbmRsZToKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGhhbmRsZS5yZWFkKDEwMjQgKiAxMDI0KSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiB3cml0ZV9jc3YocGF0aCwgZmllbGRzLCByb3dzKToKICAgIHdpdGggcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgaGFuZGxlOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGhhbmRsZSwgZmllbGRuYW1lcz1maWVsZHMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICB3cml0ZXIud3JpdGVyb3dzKHJvd3MpCgoKZGVmIHZhbGlkYXRlZF9mcmFtZXMoY2FzZSk6CiAgICBmcmFtZXMgPSBjYXNlWyJmcmFtZXMiXQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyInNhbXBsZV9cZCtfcVxkKyIsIGNhc2VbImNhc2VfaWQiXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5leHBlY3RlZCBjYXNlIElEIikKICAgIGlmIG5vdCAxIDw9IGxlbihmcmFtZXMpIDw9IDIwIG9yIFtmWyJzYW1wbGVkX2luZGV4Il0gZm9yIGYgaW4gZnJhbWVzXSAhPSBzb3J0ZWQoe2ZbInNhbXBsZWRfaW5kZXgiXSBmb3IgZiBpbiBmcmFtZXN9KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCBzYW1wbGUgb3JkZXI6IHtjYXNlWydjYXNlX2lkJ119IikKICAgIGZvciBpLCBmIGluIGVudW1lcmF0ZShmcmFtZXMpOgogICAgICAgIGV4cGVjdGVkID0gInJlY2VudCIgaWYgaSA+PSBsZW4oZnJhbWVzKSAtIDQgZWxzZSAib2xkZXIiCiAgICAgICAgaWYgKGZbInJlZ2lvbiJdICE9IGV4cGVjdGVkIG9yIGZbInNvdXJjZV9mcmFtZSJdIDwgMAogICAgICAgICAgICAgICAgb3Igbm90IDAgPD0gZlsic3RhcnRfcyJdIDw9IGNhc2VbInF1ZXJ5X3RpbWVfcyJdICsgMWUtNik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIG9yIHBvc3QtcXVlcnkgZnJhbWUgaW4ge2Nhc2VbJ2Nhc2VfaWQnXX0iKQogICAgcmV0dXJuIGZyYW1lcwoKCmRlZiByZWFkZXJfZm9yKGNhc2UsIG1lZGlhX3Jvb3QpOgogICAgZnJvbSBkZWNvcmQgaW1wb3J0IFZpZGVvUmVhZGVyLCBjcHUKCiAgICBwYXRoID0gbWVkaWFfcm9vdCAvIGYic2FtcGxlX3tjYXNlWydzYW1wbGVfaWQnXX1fcmVhbC5tcDQiCiAgICBpZiBub3QgcGF0aC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIG9yaWdpbmFsIGNsaXA6IHtwYXRofSIpCiAgICBpZiBzaGEocGF0aCkgIT0gY2FzZVsidmlkZW9fc2hhMjU2Il06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkNsaXAgY2hlY2tzdW0gZGlmZmVycyBmcm9tIHRoZSBjb25maXJtYXRpb24gcnVuOiB7cGF0aC5uYW1lfSIpCiAgICByZXR1cm4gVmlkZW9SZWFkZXIoc3RyKHBhdGgpLCBjdHg9Y3B1KDApLCBudW1fdGhyZWFkcz0xKQoKCmRlZiB2ZXJpZnlfdGltZXN0YW1wcyhyZWFkZXIsIGZyYW1lcyk6CiAgICBvcmlnaW4gPSBmbG9hdChyZWFkZXIuZ2V0X2ZyYW1lX3RpbWVzdGFtcCgwKVswXSkKICAgIGluZGljZXMgPSBbZlsic291cmNlX2ZyYW1lIl0gZm9yIGYgaW4gZnJhbWVzXQogICAgaWYgbWF4KGluZGljZXMpID49IGxlbihyZWFkZXIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNhdmVkIGZyYW1lIGluZGV4IG91dHNpZGUgZGVjb2RlZCBjbGlwIikKICAgIHRpbWVzID0gcmVhZGVyLmdldF9mcmFtZV90aW1lc3RhbXAoaW5kaWNlcykudG9saXN0KCkKICAgIGlmIGFueShhYnModFswXSAtIG9yaWdpbiAtIGZbInN0YXJ0X3MiXSkgPiAwLjAwMSBmb3IgdCwgZiBpbiB6aXAodGltZXMsIGZyYW1lcywgc3RyaWN0PVRydWUpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEZWNvZGVyIHRpbWVzdGFtcHMgZGlmZmVyIGZyb20gdGhlIGF1ZGl0ZWQgc2FtcGxpbmcgcmVjb3JkIikKCgpkZWYgcmVuZGVyX3NoZWV0KHJlYWRlciwgZnJhbWVzLCB0aXRsZSwgY29sdW1ucywgd2lkdGgsIGhlaWdodCwgb3V0cHV0KToKICAgIGZyb20gUElMIGltcG9ydCBJbWFnZSwgSW1hZ2VEcmF3LCBJbWFnZUZvbnQKCiAgICBtYXJnaW4sIGNhcHRpb24gPSAxMCwgMjcKICAgIHZlcnRpY2FsID0gKGxlbihmcmFtZXMpICsgY29sdW1ucyAtIDEpIC8vIGNvbHVtbnMKICAgIGNhbnZhcyA9IEltYWdlLm5ldygiUkdCIiwgKGNvbHVtbnMqKHdpZHRoICsgbWFyZ2luKSttYXJnaW4sIDQ4K3ZlcnRpY2FsKihoZWlnaHQrY2FwdGlvbittYXJnaW4pKSwgIiNmYWZhZmEiKQogICAgZHJhdyA9IEltYWdlRHJhdy5EcmF3KGNhbnZhcykKICAgIHRyeToKICAgICAgICBmb250ID0gSW1hZ2VGb250LnRydWV0eXBlKCIvdXNyL3NoYXJlL2ZvbnRzL3RydWV0eXBlL2RlamF2dS9EZWphVnVTYW5zLnR0ZiIsIDE1KQogICAgICAgIHRpdGxlX2ZvbnQgPSBJbWFnZUZvbnQudHJ1ZXR5cGUoIi91c3Ivc2hhcmUvZm9udHMvdHJ1ZXR5cGUvZGVqYXZ1L0RlamFWdVNhbnMtQm9sZC50dGYiLCAxOCkKICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgIGZvbnQgPSB0aXRsZV9mb250ID0gSW1hZ2VGb250LmxvYWRfZGVmYXVsdCgpCiAgICBkcmF3LnRleHQoKG1hcmdpbiwgMTIpLCB0aXRsZSwgYW5jaG9yPSJsdCIsIGZpbGw9IiMxODIzMzQiLCBmb250PXRpdGxlX2ZvbnQpCiAgICBmb3IgaSwgZnJhbWUgaW4gZW51bWVyYXRlKGZyYW1lcyk6CiAgICAgICAgeCA9IG1hcmdpbiArIChpICUgY29sdW1ucykqKHdpZHRoK21hcmdpbikKICAgICAgICB5ID0gNDIgKyAoaSAvLyBjb2x1bW5zKSooaGVpZ2h0K2NhcHRpb24rbWFyZ2luKQogICAgICAgIGltYWdlID0gSW1hZ2UuZnJvbWFycmF5KHJlYWRlcltmcmFtZVsic291cmNlX2ZyYW1lIl1dLmFzbnVtcHkoKSkKICAgICAgICBpbWFnZS50aHVtYm5haWwoKHdpZHRoLCBoZWlnaHQpLCBJbWFnZS5SZXNhbXBsaW5nLkxBTkNaT1MpCiAgICAgICAgZHJhdy5yZWN0YW5nbGUoKHgsIHksIHgrd2lkdGgsIHkraGVpZ2h0KSwgZmlsbD0iIzIyMjIyMiIpCiAgICAgICAgY2FudmFzLnBhc3RlKGltYWdlLCAoeCsod2lkdGgtaW1hZ2Uud2lkdGgpLy8yLCB5KyhoZWlnaHQtaW1hZ2UuaGVpZ2h0KS8vMikpCiAgICAgICAgdGV4dCA9IGYiU2FtcGxlIHtmcmFtZVsnc2FtcGxlZF9pbmRleCddfSAgICB7ZnJhbWVbJ3N0YXJ0X3MnXTouMmZ9IHMiCiAgICAgICAgZHJhdy50ZXh0KCh4LCB5K2hlaWdodCs1KSwgdGV4dCwgYW5jaG9yPSJsdCIsIGZpbGw9IiMxODIzMzQiLCBmb250PWZvbnQpCiAgICBjYW52YXMuc2F2ZShvdXRwdXQsIHF1YWxpdHk9OTIpCgoKZGVmIHJlbmRlcihjYXNlLCByZWFkZXIsIG91dHB1dCk6CiAgICBmcmFtZXMgPSB2YWxpZGF0ZWRfZnJhbWVzKGNhc2UpCiAgICB2ZXJpZnlfdGltZXN0YW1wcyhyZWFkZXIsIGZyYW1lcykKICAgIGZvbGRlciA9IG91dHB1dCAvICJzaGVldHMiIC8gY2FzZVsiY2FzZV9pZCJdCiAgICBmb2xkZXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmVjZW50ID0gW2YgZm9yIGYgaW4gZnJhbWVzIGlmIGZbInJlZ2lvbiJdID09ICJyZWNlbnQiXQogICAgb2xkZXIgPSBbZiBmb3IgZiBpbiBmcmFtZXMgaWYgZlsicmVnaW9uIl0gPT0gIm9sZGVyIl0KICAgIHRpdGxlID0gZiJ7Y2FzZVsnY2FzZV9pZCddfSAgfCAgUXVlc3Rpb24gYXQge2Nhc2VbJ3F1ZXJ5X3RpbWVfcyddOi4yZn0gcyIKICAgIHJlbmRlcl9zaGVldChyZWFkZXIsIHJlY2VudCwgdGl0bGUgKyAiICB8ICBSZWNlbnQgZnJhbWVzIiwgMiwgNTYwLCAzMTUsIGZvbGRlci8icmVjZW50LmpwZyIpCiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4ob2xkZXIpLCA4KToKICAgICAgICByZW5kZXJfc2hlZXQocmVhZGVyLCBvbGRlcltpOmkrOF0sIHRpdGxlICsgZiIgIHwgIE9sZGVyIGZyYW1lcywgcGFnZSB7aS8vOCsxfSIsCiAgICAgICAgICAgICAgICAgICAgIDQsIDI5NSwgMTY2LCBmb2xkZXIvZiJvbGRlcl97aS8vOCsxfS5qcGciKQoKCmRlZiBibGluZF9pbmRleChjYXNlcywgb3V0cHV0KToKICAgIGxpbmVzID0gWwogICAgICAgICc8IWRvY3R5cGUgaHRtbD48bWV0YSBjaGFyc2V0PSJ1dGYtOCI+PHRpdGxlPkhFUk1FUyB2aXN1YWwgZXZpZGVuY2UgcmV2aWV3PC90aXRsZT4nLAogICAgICAgICI8aDE+SEVSTUVTIHZpc3VhbCBldmlkZW5jZSByZXZpZXc8L2gxPiIsCiAgICAgICAgIjxwPlJldmlldyB0aGUgcXVlc3Rpb24gYW5kIGZyYW1lcyBiZWZvcmUgY29uc3VsdGluZyB0aGUgc2VwYXJhdGUgYW5zd2VyIGtleS4gIgogICAgICAgICJTYW1wbGUgbnVtYmVycyByZWZlciB0byB0aGUgb3JpZ2luYWwgMC41IEZQUyBncmlkLjwvcD4iLAogICAgXQogICAgZm9yIGNhc2UgaW4gY2FzZXM6CiAgICAgICAga2V5ID0gY2FzZVsiY2FzZV9pZCJdCiAgICAgICAgZm9sZGVyID0gb3V0cHV0IC8gInNoZWV0cyIgLyBrZXkKICAgICAgICBsaW5lcy5leHRlbmQoW2YnPGgyIGlkPSJ7a2V5fSI+e2tleX0gwrcge2Nhc2VbInF1ZXJ5X3RpbWVfcyJdOi4yZn0gczwvaDI+JywKICAgICAgICAgICAgICAgICAgICAgIGYiPHA+e2h0bWwuZXNjYXBlKGNhc2VbJ3F1ZXN0aW9uJ10pfTwvcD48b2wgdHlwZT0nQSc+Il0pCiAgICAgICAgbGluZXMuZXh0ZW5kKGYiPGxpPntodG1sLmVzY2FwZShjaG9pY2UpfTwvbGk+IiBmb3IgY2hvaWNlIGluIGNhc2VbImNob2ljZXMiXSkKICAgICAgICBsaW5lcy5hcHBlbmQoIjwvb2w+IikKICAgICAgICBmb3IgZmlsZSBpbiBbZm9sZGVyLyJyZWNlbnQuanBnIiwgKnNvcnRlZChmb2xkZXIuZ2xvYigib2xkZXJfKi5qcGciKSldOgogICAgICAgICAgICBwYXRoID0gZmlsZS5yZWxhdGl2ZV90byhvdXRwdXQpLmFzX3Bvc2l4KCkKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYnPGltZyBzcmM9IntwYXRofSIgYWx0PSJ7ZmlsZS5zdGVtfSIgc3R5bGU9ImRpc3BsYXk6YmxvY2s7bWF4LXdpZHRoOjEwMCU7bWFyZ2luOjEycHggMCI+JykKICAgIChvdXRwdXQgLyAiaW5kZXguaHRtbCIpLndyaXRlX3RleHQoIlxuIi5qb2luKGxpbmVzKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCgoKZGVmIGdlbmVyYXRlKHJlZmVyZW5jZSwgbWVkaWEsIG91dHB1dCk6CiAgICBjYXNlcyA9IHJlZmVyZW5jZVsiY2FzZXMiXQogICAgb3V0Y29tZSA9IHJlZmVyZW5jZVsib3V0Y29tZXMiXQogICAgaWYgbGVuKGNhc2VzKSAhPSAxMiBvciB7Y1siY2FzZV9pZCJdIGZvciBjIGluIGNhc2VzfSAhPSB7clsiY2FzZV9pZCJdIGZvciByIGluIG91dGNvbWV9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgZGlzY29yZGFuY2UgbGlzdCIpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgYnlfY2xpcCA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3IgY2FzZSBpbiBjYXNlczoKICAgICAgICB2YWxpZGF0ZWRfZnJhbWVzKGNhc2UpCiAgICAgICAgYnlfY2xpcFtjYXNlWyJzYW1wbGVfaWQiXV0uYXBwZW5kKGNhc2UpCiAgICBmb3Igc2lkLCBncm91cCBpbiBieV9jbGlwLml0ZW1zKCk6CiAgICAgICAgaWYgbGVuKHtjWyJ2aWRlb19zaGEyNTYiXSBmb3IgYyBpbiBncm91cH0pICE9IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDb25mbGljdGluZyBtZWRpYSBjaGVja3N1bXMgZm9yIHNhbXBsZSB7c2lkfSIpCiAgICAgICAgcmVhZGVyID0gcmVhZGVyX2Zvcihncm91cFswXSwgbWVkaWEpCiAgICAgICAgZm9yIGNhc2UgaW4gZ3JvdXA6CiAgICAgICAgICAgIHJlbmRlcihjYXNlLCByZWFkZXIsIG91dHB1dCkKICAgICAgICAgICAgcHJpbnQoZiJWSVNVQUxfQ0FTRV9SRUFEWSB7Y2FzZVsnY2FzZV9pZCddfSIsIGZsdXNoPVRydWUpCiAgICAgICAgZGVsIHJlYWRlcgogICAgYmxpbmRfaW5kZXgoY2FzZXMsIG91dHB1dCkKICAgICMgU2F2ZSBleGlzdGluZyBqdWRnZW1lbnRzIGFjcm9zcyBub3RlYm9vayByZXJ1bnMuCiAgICBub3RlcyA9IG91dHB1dCAvICJyZXZpZXdfbm90ZXMuY3N2IgogICAgaWYgbm90IG5vdGVzLmV4aXN0cygpOgogICAgICAgIHdyaXRlX2Nzdihub3RlcywgTk9URVNfRklFTERTLCBbeyJjYXNlX2lkIjogY1siY2FzZV9pZCJdfSBmb3IgYyBpbiBjYXNlc10pCiAgICBlbHNlOgogICAgICAgIHdpdGggbm90ZXMub3BlbihlbmNvZGluZz0idXRmLTgtc2lnIiwgbmV3bGluZT0iIikgYXMgaGFuZGxlOgogICAgICAgICAgICBzYXZlZCA9IGNzdi5EaWN0UmVhZGVyKGhhbmRsZSkKICAgICAgICAgICAgaWYgdHVwbGUoc2F2ZWQuZmllbGRuYW1lcyBvciAoKSkgIT0gTk9URVNfRklFTERTIG9yIFtyWyJjYXNlX2lkIl0gZm9yIHIgaW4gc2F2ZWRdICE9IFtjWyJjYXNlX2lkIl0gZm9yIGMgaW4gY2FzZXNdOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhpc3RpbmcgcmV2aWV3IG5vdGVzIGRvIG5vdCBtYXRjaCB0aGUgZnJvemVuIGNhc2VzIikKICAgIGtleSA9IG91dHB1dCAvICJhZnRlcl9yZXZpZXciIC8gIm91dGNvbWVfa2V5LmNzdiIKICAgIGtleS5wYXJlbnQubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIHdyaXRlX2NzdihrZXksIHR1cGxlKG91dGNvbWVbMF0pLCBvdXRjb21lKQogICAgd3JpdGVfY3N2KG91dHB1dCAvICJmcmFtZV9zbG90cy5jc3YiLCAoImNhc2VfaWQiLCAic2FtcGxlZF9pbmRleCIsICJzb3VyY2VfZnJhbWUiLCAic3RhcnRfcyIsICJyZWdpb24iKSwgWwogICAgICAgIGRpY3QoY2FzZV9pZD1jWyJjYXNlX2lkIl0sICoqZikgZm9yIGMgaW4gY2FzZXMgZm9yIGYgaW4gY1siZnJhbWVzIl0KICAgIF0pCiAgICAob3V0cHV0IC8gInJldmlld19yZWZlcmVuY2UuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KGFyY2hpdmVfc2hhMjU2PXJlZmVyZW5jZVsiYXJjaGl2ZV9zaGEyNTYiXSwKICAgICAgICBjYXNlcz1jYXNlcywgc3VwcG9ydF9yZWdpb25zPVJFR0lPTlMpLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCgoKZGVmIGluc3BlY3RfZnJhbWUocmVmZXJlbmNlLCBtZWRpYSwgb3V0cHV0LCBjYXNlX2lkLCBzYW1wbGVkX2luZGV4KToKICAgIGZyb20gUElMIGltcG9ydCBJbWFnZQoKICAgIGNhc2UgPSBuZXh0KChjIGZvciBjIGluIHJlZmVyZW5jZVsiY2FzZXMiXSBpZiBjWyJjYXNlX2lkIl0gPT0gY2FzZV9pZCksIE5vbmUpCiAgICBpZiBjYXNlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5rbm93biBjYXNlIElEIikKICAgIGZyYW1lID0gbmV4dCgoZiBmb3IgZiBpbiB2YWxpZGF0ZWRfZnJhbWVzKGNhc2UpIGlmIGZbInNhbXBsZWRfaW5kZXgiXSA9PSBzYW1wbGVkX2luZGV4KSwgTm9uZSkKICAgIGlmIGZyYW1lIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVxdWVzdGVkIHNhbXBsZSB3YXMgb3V0c2lkZSB0aGUgbW9kZWwgaW5wdXQiKQogICAgcmVhZGVyID0gcmVhZGVyX2ZvcihjYXNlLCBtZWRpYSkKICAgIHZlcmlmeV90aW1lc3RhbXBzKHJlYWRlciwgW2ZyYW1lXSkKICAgIGZvbGRlciA9IG91dHB1dCAvICJvcmlnaW5hbHMiCiAgICBmb2xkZXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aCA9IGZvbGRlciAvIGYie2Nhc2VfaWR9X3NhbXBsZV97c2FtcGxlZF9pbmRleDowNGR9LnBuZyIKICAgIEltYWdlLmZyb21hcnJheShyZWFkZXJbZnJhbWVbInNvdXJjZV9mcmFtZSJdXS5hc251bXB5KCkpLnNhdmUocGF0aCkKICAgIHByaW50KHBhdGgpCgoKZGVmIHBhY2thZ2Uob3V0cHV0LCBkZXN0KToKICAgIGZpbGVzID0gW291dHB1dC9uIGZvciBuIGluICgiaW5kZXguaHRtbCIsICJyZXZpZXdfbm90ZXMuY3N2IiwgImZyYW1lX3Nsb3RzLmNzdiIsICJyZXZpZXdfcmVmZXJlbmNlLmpzb24iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImFmdGVyX3Jldmlldy9vdXRjb21lX2tleS5jc3YiKV0KICAgIGZpbGVzICs9IHNvcnRlZCgob3V0cHV0IC8gInNoZWV0cyIpLmdsb2IoIiovKi5qcGciKSkKICAgIGZpbGVzICs9IHNvcnRlZCgob3V0cHV0IC8gIm9yaWdpbmFscyIpLmdsb2IoIioucG5nIikpCiAgICB0ZW1wID0gZGVzdC53aXRoX3N1ZmZpeCgiLnRtcCIpCiAgICB3aXRoIFppcEZpbGUodGVtcCwgInciLCBjb21wcmVzc2lvbj1aSVBfREVGTEFURUQpIGFzIHppcHBlZDoKICAgICAgICBmb3IgcGF0aCBpbiBmaWxlczoKICAgICAgICAgICAgemlwcGVkLndyaXRlKHBhdGgsIHBhdGgucmVsYXRpdmVfdG8ob3V0cHV0KS5hc19wb3NpeCgpKQogICAgdGVtcC5yZXBsYWNlKGRlc3QpCiAgICBwcmludChmIlZJU1VBTF9SRVZJRVdfWklQIHtkZXN0fSBTSEEyNTY9e3NoYShkZXN0KX0iLCBmbHVzaD1UcnVlKQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgZm9yIG4gaW4gKCJyZWZlcmVuY2UiLCAibWVkaWEiLCAib3V0cHV0Iik6CiAgICAgICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS0iK24sIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taW5zcGVjdCIsIG5hcmdzPTIsIG1ldGF2YXI9KCJDQVNFIiwgIlNBTVBMRSIpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS16aXAiLCB0eXBlPVBhdGgpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmVmZXJlbmNlID0ganNvbi5sb2FkcyhhcmdzLnJlZmVyZW5jZS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBpZiBhcmdzLmluc3BlY3Q6CiAgICAgICAgaW5zcGVjdF9mcmFtZShyZWZlcmVuY2UsIGFyZ3MubWVkaWEsIGFyZ3Mub3V0cHV0LCBhcmdzLmluc3BlY3RbMF0sIGludChhcmdzLmluc3BlY3RbMV0pKQogICAgZWxpZiBhcmdzLnppcDoKICAgICAgICBwYWNrYWdlKGFyZ3Mub3V0cHV0LCBhcmdzLnppcCkKICAgIGVsc2U6CiAgICAgICAgZ2VuZXJhdGUocmVmZXJlbmNlLCBhcmdzLm1lZGlhLCBhcmdzLm91dHB1dCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==', '06cefac820e93074c221dd307de324bfd6984e3e40eea32beda20b7dff57d81c')}
for relative, (encoded, digest) in PAYLOAD.items():
    payload = base64.b64decode(encoded, validate=True)
    if hashlib.sha256(payload).hexdigest() != digest:
        raise ValueError(f'Embedded review input is corrupt: {relative}')
    if relative.endswith('.py'):
        compile(payload, relative, 'exec')
    if not VALIDATE_ONLY:
        target = WORK / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.read_bytes() != payload:
            raise RuntimeError(f'Review input changed: {target}. Use a new WORK directory.')
        if not target.exists():
            target.write_bytes(payload)

if VALIDATE_ONLY:
    print('BLIND_REVIEW_VALIDATION_PASS')
else:
    from IPython.display import FileLink, display

    python_choices = (
        BASE / 'hermes_target_binding/venv/bin/python',
        BASE / 'hermes_causal_pilot/venv/bin/python',
        Path(sys.executable),
    )
    python = next((p for p in python_choices if p.is_file() and subprocess.run(
        [str(p), '-c', 'import decord, PIL'], stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL).returncode == 0), None)
    if python is None:
        raise RuntimeError('The experiment Python with Decord and Pillow is missing.')

    artifact = WORK / OUTPUT_NAME
    worksheet = WORK / 'visual_support_review.csv'
    if not artifact.exists():
        command = [str(python), '-u', str(WORK / 'render_target_binding_review.py'),
                   '--worksheet', str(worksheet), '--evidence', str(WORK / 'review_reference.zip'),
                   '--videos', str(MEDIA), '--output', str(artifact)]
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line.rstrip(), flush=True)
            if proc.wait():
                raise RuntimeError('Review rendering failed. See the clip or environment error above.')
    with ZipFile(artifact) as rendered:
        with worksheet.open(encoding='utf-8', newline='') as handle:
            ids = {row['review_id'] for row in csv.DictReader(handle)}
        expected = {'review.html', worksheet.name} | {f'frames/{rid}.jpg' for rid in ids}
        if (len(ids) != 147 or set(rendered.namelist()) != expected or rendered.testzip() is not None
                or rendered.read(worksheet.name) != worksheet.read_bytes()):
            raise ValueError('The rendered ZIP differs from the blinded worksheet. Use a new WORK directory.')
    link = Path.cwd() / OUTPUT_NAME
    if link.resolve() != artifact.resolve():
        if link.exists():
            if hashlib.sha256(link.read_bytes()).digest() != hashlib.sha256(artifact.read_bytes()).digest():
                raise FileExistsError(f'Choose a clean notebook directory. A different packet exists at {link}.')
        else:
            shutil.copyfile(artifact, link)
    print('BLIND_REVIEW_PACKET_READY | 147 windows | 32 questions')
    display(FileLink(link.name))
